# T1 overlap (P7 secondary analysis S6), 23 Oct 2026

Registered as S6 in P7, `prereg\PREREG_verifier_ceiling.md`. The registration is in the public repo,
commit 7a7856f, pushed 23 Oct 2026 at 04:45 IST. Its SHA-256 is f6dfbdd8…. B3 lets S6 run once P7 is
committed, before the Phase 0 gates.

**S6 as registered:** per dataset at k≥3 over m4, the certified-wrong problems shared by all three
generators, by each pair only, and by one generator only, with their pids. Descriptive.

- Inputs: the certificate files and `v2_reverified_rows.json` (B3). GSM-Hard is also scored at the gold's
  stored precision (S7), which needs its three generator files.
- Answers read with extract_final_v2; as-run (v1) values are shown beside them where they differ (B0).
- Datasets: SVAMP, GSM8K, GSM-Hard, GSM-Symbolic main and p1. Main and p1 have template 76 excluded, and
  are also reported unfiltered (B0). p2 is out of scope.
- Nothing else from Phase 1 is computed here: no consensus sets, no vetoes, no λ.
- Gates first. Each must reproduce numbers already seen before registration (B1, G0.1), or the notebook stops.
- CPU only, no Z3 calls. ckpt\ is read only. Writes only `out\verifier_ceiling\<timestamp>_S6\`.
- Run from `Videos\tost flatness`, or set the paths in the first code cell.

In [ ]:
# =============================================================================
# CELL 1 - PATHS, SETTINGS AND THE REGISTRATION GATE
# =============================================================================
import csv, datetime as dt, hashlib, importlib.util, json, sys
from fractions import Fraction as F
from pathlib import Path

ROOT = Path.cwd()                                    # Videos\tost flatness
CKPT = ROOT / "ckpt"
STAGE2 = ROOT / "stage2_lib.py"                      # the as-run copy (to_fraction and answers_match only)
V2 = ROOT / "extract_final_v2.py"
MSP = ROOT / "match_stored_precision.py"
PREREG = ROOT / "prereg" / "PREREG_verifier_ceiling.md"
RERUN_DIR = ROOT / "out" / "rerun_v2" / "20261023_014600"     # the laptop run of rerun_with_v2.ipynb
FLASHLITE_DIR = Path(r"C:\Users\Abhis\Videos\Audit\out\google_api_audit\20260917_162638")

P7_SHA256_LF = "f6dfbdd8f03a7677011351893927900b5f60662dc1bd0d243df7fce7717f3a15"
AS_RUN_SHA256_LF = "ae042d0c48c9d0f088ade4fc718ba5b56fc37328c9b0e47ca1306f2c20941388"
DATASETS = ["svamp", "gsm8k", "gsm_hard", "gsm_symbolic_main", "gsm_symbolic_p1"]
SYMBOLIC = {"gsm_symbolic_main", "gsm_symbolic_p1"}
EXCLUDED_TEMPLATE = 76
GENS = ["qwen", "flashlite", "pro"]
M4, K = ["t1", "t2", "t3", "t6"], 3
N_EXPECTED = {"svamp": 1000, "gsm8k": 1319, "gsm_hard": 1319, "gsm_symbolic_main": 5000, "gsm_symbolic_p1": 5000}

for p in (CKPT, STAGE2, V2, MSP, PREREG, RERUN_DIR / "v2_reverified_rows.json", FLASHLITE_DIR):
    assert p.exists(), f"not found: {p}"


def sha256_lf(path):
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read().replace(b"\r\n", b"\n")).hexdigest()


assert sha256_lf(PREREG) == P7_SHA256_LF, "REGISTRATION GATE FAILED: prereg file is not the registered P7 text"
RUN_TS = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "out" / "verifier_ceiling" / f"{RUN_TS}_S6"
assert CKPT.resolve() not in OUT.resolve().parents
print("root:", ROOT, "\noutputs:", OUT)
print("REGISTRATION GATE passed: P7 present with the registered SHA-256", P7_SHA256_LF[:16] + "…")

In [ ]:
# =============================================================================
# CELL 2 - MODULES (no solver calls) AND READERS
# =============================================================================
def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


assert sha256_lf(STAGE2) == AS_RUN_SHA256_LF, f"{STAGE2} is not the as-run stage2_lib.py"
s2 = load_module(STAGE2, "stage2_lib_as_run")
v2 = load_module(V2, "extract_final_v2")
msp = load_module(MSP, "match_stored_precision")
print(v2.VERSION, "|", msp.VERSION, f"(self-test {msp._self_test()} cases ok)", "| python", sys.version.split()[0])


def read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def verdict(v):
    """Bools, except Flash-Lite rows verified on 17 Sep, which store verify()'s dict."""
    if isinstance(v, bool):
        return v
    if isinstance(v, dict):
        return bool(v.get("certified", False))
    raise TypeError(repr(v)[:80])


def template_of(pid):
    tail = pid.split("/")[1]
    return int(tail.split("_")[0]) if "_" in tail else None


def cert_block(gen, ds):
    if gen == "flashlite":
        return read_json(FLASHLITE_DIR / f"w2s_cert_flashlite_{ds}.json")["flashlite"]
    if ds in SYMBOLIC:
        return read_json(CKPT / f"w2s_cert_{ds}.json")["qwen" if gen == "qwen" else "gemini"]
    d = read_json(CKPT / (f"w2s_cert_{ds}.json" if gen == "pro" else f"_verdict_cache_{ds}.json"))
    return {"correct": d["correct"], "cert": d["cert"]}

In [ ]:
# =============================================================================
# CELL 3 - PER-ROW STATE FROM THE CERTIFICATE FILES (v1) PLUS THE 107 RE-VERIFIED ROWS (v2)
# =============================================================================
REV = read_json(RERUN_DIR / "v2_reverified_rows.json")["changed_rows"]
assert len(REV) == 107, len(REV)
STATE = {}
for ds in DATASETS:
    for gen in GENS:
        blk = cert_block(gen, ds)
        pids = sorted(blk["correct"])
        assert len(pids) == N_EXPECTED[ds], (gen, ds, len(pids))          # the loader is the invariant
        assert all(set(blk["cert"][t]) == set(pids) for t in M4), (gen, ds, "cert pids")
        STATE[(gen, ds)] = {p: {"ok1": bool(blk["correct"][p]), "cert1": {t: verdict(blk["cert"][t][p]) for t in M4},
                                "eligible": ds not in SYMBOLIC or template_of(p) != EXCLUDED_TEMPLATE}
                            for p in pids}
n_rev = 0
for r in REV:
    if r["dataset"] not in DATASETS:
        continue                                        # p2 rows: out of scope
    s = STATE[(r["generator"], r["dataset"])][r["pid"]]
    assert s["ok1"] == r["correct_v1"] and all(s["cert1"][t] == r["cert_v1"][t] for t in M4), ("rev mismatch", r["pid"])
    s["ok2"], s["cert2"] = r["correct_v2"], {t: r["cert_v2"][t] for t in M4}
    n_rev += 1
for rows in STATE.values():
    for s in rows.values():
        s.setdefault("ok2", s["ok1"])
        s.setdefault("cert2", s["cert1"])
print(f"{n_rev} re-verified rows applied (their stored v1 verdicts and scores match the certificate files)")


def cw_set(gen, ds, v, filtered=True, ok_key=None):
    """Certified-wrong pids at k>=3 over m4. ok_key overrides the scorer (S7)."""
    out = set()
    for p, s in STATE[(gen, ds)].items():
        if filtered and not s["eligible"]:
            continue
        if sum(s[f"cert{v}"][t] for t in M4) >= K and not s[ok_key or f"ok{v}"]:
            out.add(p)
    return out


def accepted(gen, ds, v, filtered=True):
    return sum(1 for s in STATE[(gen, ds)].values()
               if (s["eligible"] or not filtered) and sum(s[f"cert{v}"][t] for t in M4) >= K)

In [ ]:
# =============================================================================
# CELL 4 - GATES. Every count here was seen before registration (B1, G0.1).
# =============================================================================
# G1: v1 reproduces the published k>=3 m4 cells of arms_three_generators.json (A and W).
ARMS = read_json(FLASHLITE_DIR / "arms_three_generators.json")
GEN_KEY = {"qwen": "qwen", "flashlite": "flashlite", "pro": "gemini"}
bad = [(ds, g) for ds in DATASETS for g in GENS
       if (accepted(g, ds, 1), len(cw_set(g, ds, 1))) != (ARMS[ds][f"k>={K} m4"][GEN_KEY[g]]["A"], ARMS[ds][f"k>={K} m4"][GEN_KEY[g]]["W"])]
assert not bad, f"GATE 1 FAILED: {bad}"
print("GATE 1 passed: v1 reproduces the published k>=3 m4 A and W for 3 generators x 5 datasets")

# G2: v2 reproduces P7's G0.1 headline cells (A / W at k>=3), unfiltered main included.
G01 = {("qwen", "svamp"): (560, 4), ("qwen", "gsm8k"): (467, 7), ("qwen", "gsm_hard"): (320, 26),
       ("qwen", "gsm_symbolic_main"): (1710, 0), ("qwen", "gsm_symbolic_p1"): (845, 0),
       ("pro", "svamp"): (577, 5), ("pro", "gsm8k"): (475, 7), ("pro", "gsm_hard"): (421, 41),
       ("pro", "gsm_symbolic_main"): (1788, 0), ("pro", "gsm_symbolic_p1"): (925, 0),
       ("flashlite", "gsm_hard"): (383, 29), ("flashlite", "gsm_symbolic_main"): (1786, 0),
       ("flashlite", "gsm_symbolic_p1"): (924, 0)}
bad = [(k_, (accepted(*k_, 2), len(cw_set(*k_, 2))), want) for k_, want in G01.items()
       if (accepted(*k_, 2), len(cw_set(*k_, 2))) != want]
for g, want in (("qwen", (1749, 28)), ("pro", (1828, 29))):
    got = (accepted(g, "gsm_symbolic_main", 2, filtered=False), len(cw_set(g, "gsm_symbolic_main", 2, filtered=False)))
    if got != want:
        bad.append(((g, "main unfiltered"), got, want))
assert not bad, f"GATE 2 FAILED: {bad}"
print("GATE 2 passed: v2 reproduces P7's G0.1 cells, including unfiltered main (Qwen 1,749/28, Pro 1,828/29)")

# G3: Qwen vs Pro alone reproduces the transition counts in B1 (seen 27 Sep and 22 Oct).
SEEN = {"svamp": (4, 1, 0), "gsm8k": (5, 2, 2), "gsm_hard": (23, 18, 3)}    # shared, Pro-only, Qwen-only (v2)
for ds, want in SEEN.items():
    q, p = cw_set("qwen", ds, 2), cw_set("pro", ds, 2)
    assert (len(q & p), len(p - q), len(q - p)) == want, ("GATE 3 FAILED", ds, (len(q & p), len(p - q), len(q - p)), want)
q1, p1_ = cw_set("qwen", "gsm_hard", 1), cw_set("pro", "gsm_hard", 1)
assert (len(q1 & p1_), len(p1_ - q1)) == (23, 14), "GATE 3 FAILED: GSM-Hard as run"
print("GATE 3 passed: Qwen/Pro shared, Pro-only, Qwen-only = 4/1/0, 5/2/2, 23/18/3 (GSM-Hard as run 23/14)")

# G4 (for S7): GSM-Hard answers re-read with v2 reproduce the stored scores, and the stored-precision
# certified-wrong counts reproduce the rerun (v1 27/32/40, v2 27/32/45).
GEN_FILES = {"qwen": "raw_gen_{ds}.jsonl", "flashlite": "raw_gen_flashlite_{ds}.jsonl", "pro": "raw_gen_gemini_{ds}.jsonl"}
v1_of = {(r["generator"], r["pid"]): F(r["v1"]) for r in REV if r["dataset"] == "gsm_hard"}
for g in GENS:
    recs = read_jsonl(CKPT / GEN_FILES[g].format(ds="gsm_hard"))
    assert len(recs) == 1319 and {r["pid"] for r in recs} == set(STATE[(g, "gsm_hard")]), (g, "gsm_hard rows")
    for r in recs:
        o = r.get("outputs")
        text = (o[0] if o else "") if isinstance(o, list) else (o or "")
        gold = s2.to_fraction(r.get("gold"))
        e2 = v2.extract_final_v2(text)[0]
        e1 = v1_of.get((g, r["pid"]), e2)
        s = STATE[(g, "gsm_hard")][r["pid"]]
        assert s2.answers_match(e2, gold) == s["ok2"], ("GATE 4 FAILED: v2 score", g, r["pid"])
        s["ok1_sp"] = msp.answers_match_stored(e1, gold)
        s["ok2_sp"] = msp.answers_match_stored(e2, gold)
got = {v: tuple(len(cw_set(g, "gsm_hard", v, ok_key=f"ok{v}_sp")) for g in GENS) for v in (1, 2)}
assert got == {1: (27, 32, 40), 2: (27, 32, 45)}, f"GATE 4 FAILED: {got}"
print("GATE 4 passed: GSM-Hard v2 scores reproduced; stored-precision CW v1 27/32/40, v2 27/32/45")

In [ ]:
# =============================================================================
# CELL 5 - S6: WHO SHARES WHICH CERTIFIED-WRONG PROBLEMS (k>=3 over m4)
# =============================================================================
CATS = [("all three", ("qwen", "flashlite", "pro")),
        ("Qwen + Flash-Lite only", ("qwen", "flashlite")), ("Qwen + Pro only", ("qwen", "pro")),
        ("Flash-Lite + Pro only", ("flashlite", "pro")),
        ("Qwen only", ("qwen",)), ("Flash-Lite only", ("flashlite",)), ("Pro only", ("pro",))]


def s6(sets):
    union = set().union(*sets.values())
    out = {}
    for name, members in CATS:
        out[name] = sorted(p for p in union if all(p in sets[g] for g in members)
                           and not any(p in sets[g] for g in GENS if g not in members))
    assert sum(len(v) for v in out.values()) == len(union)
    return union, out


VIEWS = []                                   # (dataset, population, reader, scorer)
for ds in DATASETS:
    VIEWS.append((ds, "filtered" if ds in SYMBOLIC else "all", 2, "primary"))
    if ds in SYMBOLIC:
        VIEWS.append((ds, "unfiltered", 2, "primary"))
    if ds == "gsm_hard":
        VIEWS.append((ds, "all", 2, "stored precision (S7)"))
RESULTS = []
for ds, pop, v, scorer in VIEWS + [(ds, pop, 1, sc) for ds, pop, _, sc in VIEWS]:
    ok_key = f"ok{v}_sp" if scorer.startswith("stored") else None
    sets = {g: cw_set(g, ds, v, filtered=(pop != "unfiltered"), ok_key=ok_key) for g in GENS}
    union, cats = s6(sets)
    RESULTS.append({"dataset": ds, "population": pop, "reader": f"v{v}", "scorer": scorer,
                    "W": {g: len(sets[g]) for g in GENS}, "union": len(union), "categories": cats})

key = lambda r: (r["dataset"], r["population"], r["scorer"])
V1 = {key(r): r for r in RESULTS if r["reader"] == "v1"}
short = lambda pids: " ".join(p.split("/")[1] for p in pids) or "-"
print("S6: certified-wrong problems at k>=3 over m4, by who shares them (reader v2; as-run v1 beside it where it differs)\n")
for r in (x for x in RESULTS if x["reader"] == "v2"):
    r1 = V1[key(r)]
    same = r1["categories"] == r["categories"]
    head = f"{r['dataset']} ({r['population']}, {r['scorer']})"
    print(f"{head}: W Qwen {r['W']['qwen']}, Flash-Lite {r['W']['flashlite']}, Pro {r['W']['pro']}; "
          f"{r['union']} distinct problems" + ("" if same else "   [v1 differs, see below]"))
    for name, _ in CATS:
        n = len(r["categories"][name])
        line = f"    {name:<24} {n:>3}"
        if r["union"]:
            line += f"  ({100 * n / r['union']:.0f}%)"
        if not same and len(r1["categories"][name]) != n:
            line += f"   v1: {len(r1['categories'][name])}"
        print(line)
    print()

In [ ]:
# =============================================================================
# CELL 6 - THE PIDS (the registered output), AND WRITE THE FILES
# =============================================================================
for r in (x for x in RESULTS if x["reader"] == "v2" and x["union"]):
    r1 = V1[key(r)]
    for rr in ([r] if r1["categories"] == r["categories"] else [r, r1]):    # v1 listed only where it differs
        print(f"{rr['dataset']} ({rr['population']}, {rr['scorer']}, reader {rr['reader']})")
        for name, _ in CATS:
            if rr["categories"][name]:
                print(f"    {name:<24} {short(rr['categories'][name])}")
        print()

OUT.mkdir(parents=True, exist_ok=False)
with open(OUT / "s6_overlap.csv", "w", encoding="utf-8", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["dataset", "population", "reader", "scorer", "category", "count", "share_of_union", "pids"])
    for r in RESULTS:
        for name, _ in CATS:
            pids = r["categories"][name]
            w.writerow([r["dataset"], r["population"], r["reader"], r["scorer"], name, len(pids),
                        f"{len(pids) / r['union']:.4f}" if r["union"] else "", " ".join(pids)])
with open(OUT / "s6_overlap.json", "w", encoding="utf-8") as fh:
    json.dump({"analysis": "P7 S6 (T1 overlap)", "registration": {"file": "prereg/PREREG_verifier_ceiling.md",
               "sha256_lf": P7_SHA256_LF, "commit": "7a7856f095f2ebd7d15a77f24e7647f07542c4ab"},
               "rule": f"k>={K} over m4", "run": RUN_TS, "python": sys.version.split()[0],
               "results": RESULTS}, fh, indent=1)
print("wrote", OUT)